# Inicialização

In [6]:
%pip install selenium pandas

import os
import platform
import shutil
import socket
import subprocess
import time

from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

port = 9222
profile = Path.home() / ".sefaz_automação_perfil"
url = 'https://plataforma.sefaz.ma.gov.br/'

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## funções

In [7]:
def _cbrowser():
    sis = platform.system()
    cand = []

    if sis == 'Windows':
        bases = [os.environ.get(v) for v in
                 ('PROGRAMFILES', 'PROGRAMFILES(X86)', 'LOCALAPPDATA')]
        for base in filter(None, bases):
            cand.append(('chrome', Path(base) / 'Google/Chrome/Application/chrome.exe'))
        for base in filter(None, bases):
                    cand.append(('edge', Path(base) / 'Microsoft/Edge/Application/msedge.exe'))
    else:
          for nome in ('google-chrome', 'google-chrome-stable', 'chromium', 'chromium-browser'):
                if (p := shutil.which(nome)):
                      cand.append(('chrome', Path(p)))
    return cand

In [8]:
def _fbrowser():
    for type, path in _cbrowser():
        if path.exists():
            return type, str(path)
    raise RuntimeError('Nenhum navegador compativel... Por favor, instale Edge ou Chrome')

In [9]:
def _openport():
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(('127.0.0.1', port)) == 0

In [13]:
def obrowser():
    type, exe = _fbrowser()
    profile.mkdir(parents = True, exist_ok = True)

    if not _openport(port):
        subprocess.Popen([
            exe,
            f'--remote-debugging-port={port}',
            f'--user-data-dir={profile}',
            '--no-first-run',
            '--no-default-browser-check',
            url
        ])
        for _ in range(30):
            if _openport(port):
                break
            time.sleep(0.5)
        else:
            raise RuntimeError('O navegador não abriu a porta a tempo')

    input(
        'O navegador foi aberto. Faça o login e acesse a plataforma.',
        'Após realizado o login, retorne à esta página e aperte ENTER para continuar',
        sep = '\n'
    )

    if type == 'edge':
        opt = EdgeOptions()
        opt.add_experimental_option('debuggerAddress', f'127.0.0.1:{port}')
        return webdriver.Edge(options = opt)

if __name__ == '__main__':
    driver = obrowser()
    print('Conectado via:', driver.current_url)

RuntimeError: Nenhum navegador compativel... Por favor, instale Edge ou Chrome

# Automatização

## Login

In [ ]:
driver = webdriver.Chrome()
driver.get('https://plataforma.sefaz.ma.gov.br/')

wait = WebDriverWait(driver, timeout = 15)

xpath = '//*[@id="loginForm"]/fieldset/div[8]/div/div/a'

wait.until(EC.presence_of_element_located((By.XPATH, xpath)))
driver.find_element(by = By.XPATH, value = xpath).click()

'''login_gov = driver.find_element(by = By.CSS_SELECTOR, value = '.botao-login-govbr')

login_gov.click()
'''
logs = driver.get_log("browser")

for log in logs:
    print(log)

cookies = driver.get_cookies()

for cookie in cookies:
    print({
        "name": cookie["name"],
        "domain": cookie["domain"],
        "secure": cookie["secure"],
        "httpOnly": cookie["httpOnly"]
    })

{'level': 'WARNING', 'message': 'https://plataforma.sefaz.ma.gov.br/334.495712f68f093a9e.js 0:164481 "[env.json] Não foi possível carregar. Usando environment.ts padrão." SyntaxError: Unexpected token \'<\', "<!DOCTYPE "... is not valid JSON', 'source': 'console-api', 'timestamp': 1795039343678}
{'name': 'TS0185eea4', 'domain': '.sso.acesso.gov.br', 'secure': False, 'httpOnly': False}
{'name': 'INGRESSCOOKIE', 'domain': '.sso.acesso.gov.br', 'secure': True, 'httpOnly': True}
{'name': 'Session_Gov_Br_Prod', 'domain': '.sso.acesso.gov.br', 'secure': True, 'httpOnly': True}
{'name': 'TS0197b850', 'domain': 'sso.acesso.gov.br', 'secure': False, 'httpOnly': False}
{'name': 'TS4666e2e5027', 'domain': 'sso.acesso.gov.br', 'secure': False, 'httpOnly': False}


In [ ]:
async with pw() as p:

    # Entrar na Sefaz
    ff = await p.firefox.launch_persistent_context(user_data_dir = './meu_perfil', headless = False, slow_mo = 50)
    
    page = await ff.new_page()
    await page.goto()

    await page.locator('.botao-login-govbr').click() # Clicar em "Entrar com Gov.br"

    # Login Gov

    '''
    usrn_input = page.locator('#accountId')
    await usrn_input.wait_for(state = 'visible')
    await usrn_input.fill(cpf)

    await page.get_by_text('Continuar').click()

    print('Resolva o captcha, por favor!')
    input('Clique ENTER neste caderno após cumprida a verificação contra robô...')

    psw_input = page.locator('#password-eye')
    await psw_input.wait_for(state = 'visible')
    await psw_input.fill(senha)

    await page.get_by_text('Entrar').click()
    '''
    await page.goto('https://plataforma.sefaz.ma.gov.br/#/')

    await page.get_by_text('Documentos')

    input('Pressione ENTER para fechar...')
